# Comparing Different UQ Methods

# Import

In [ ]:
import os
import dill
import numpy as np
import sys
import pathlib
import pandas as pd
import pickle
import time
from collections import defaultdict
import sys
import scipy.special

# importing modules/libs for plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import plotly.offline as pyo
# Set notebook mode to work in offline
pyo.init_notebook_mode()
import matplotlib.pyplot as plt
pd.options.plotting.backend = "plotly"

import chaospy as cp
import uqef


In [ ]:
from uqef_dynamic.utils import utility
from uqef_dynamic.utils import uqef_dynamic_utils
from uqef_dynamic.utils import create_stat_object

In [ ]:
COLORS = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#637939', '#393b79', '#8c6d31', '#843c39', '#7b4173',
    '#3182bd', '#6baed6', '#9ecae1', '#c6dbef', '#e6550d',
    '#fd8d3c', '#fdae6b', '#fdd0a2', '#31a354', '#74c476',
    '#a1d99b', '#c7e9c0', '#756bb1', '#9e9ac8', '#bcbddc'
]

PARAMETERS = ["TT", "C0", "ETF", "FC", "beta", "FRAC", "K2", "LP", "K1", "alpha"]
PARAMETERS = ["TT", "C0", "ETF", "FC", "beta", "FRAC", "K2", "LP", "K1", "alpha", "PM"]

COLORS_DICT = {PARAMETERS[idx]:COLORS[idx] for idx in range(len(PARAMETERS))}
COLORS_DICT

# Utility Functions

In [ ]:
import re

pattern = re.compile(
    r"^generalized_sobol_total_index_(?P<paramname>[^_]+)(?:_(?P<timestamp>[^_]+))?$"
)


def extract_paramname(whole_string, prefix="generalized_sobol_total_index"):
    # Regular expression pattern to match both cases
    pattern = r"generalized_sobol_total_index_(.*?)(?:_\d+)?$"
    # pattern = rf"{re.escape(prefix)}_(.*?)(?:_\d+)?$"
    
    match = re.search(pattern, whole_string)
    
    if match:
        return match.group(1)  # Extract the paramname
    return None  # Return None if no match is found


def extract_param_name_from_column_generlized_index(column_name):
    match = pattern.match(column_name)
    if match:
        paramname = match.group("paramname")
        # print(f"'{s}' --> paramname: '{paramname}'")
    else:
        paramname = None
        # print(f"'{s}' does not match the pattern")
    return paramname


def identify_column_generlized_index_form(column_name):
    match = pattern.fullmatch(column_name)
    if not match:
        # return "invalid", None
        return None, None
    paramname = match.group("paramname")
    look_back_window_size = match.group("timestamp")
    if look_back_window_size is None:
        # return "whole", paramname  # generalized_sobol_total_index_{paramname}
        return None, paramname
    else:
        return look_back_window_size, paramname  # generalized_sobol_total_index_{paramname}_{timestamp}


def _add_forcing_data(fig, df_temp):
    fig.add_trace(
        go.Scatter(
            x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp['temperature'],
            text=df_temp['temperature'],
            name="Temperature", mode='lines+markers',
            showlegend=False,
            marker_color='blue',
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp['precipitation'],
            text=df_temp['precipitation'],
            name="Precipitation",
            showlegend=False,
            line=dict(color='#CC79A7')
            # marker_color='red',
            # mode="lines",
            #         line=dict(
            #             color='LightSkyBlue')
        ),
        row=2, col=1
    )
    fig.update_yaxes(autorange="reversed", row=2, col=1)
    return fig


def _add_sensitivity_indices_as_time_signals(
    fig, df, substring="generalized_sobol_total_index_", plot_generalized=True, look_back_window_size=None, 
    color_dict=COLORS_DICT, row=1, col=1, showlegend=False):
    for single_column in df.columns:
        if single_column!=utility.TIME_COLUMN_NAME and single_column!='qoi' and single_column!='measured' and single_column.startswith(substring):
            plot_this_column = False
            if plot_generalized:
                current_look_back_window_size, current_parameter_name = identify_column_generlized_index_form(single_column)
                if look_back_window_size is not None:
                    if current_look_back_window_size is not None and int(current_look_back_window_size) == look_back_window_size:
                        plot_this_column = True
                else:
                    plot_this_column = True
            else:
                current_parameter_name = single_column.split(substring, 1)[1]  # TODO Maybe this does not hold always!
                plot_this_column = True
            
            if plot_this_column:
                fig.add_trace(
                    go.Scatter(
                        x=df[utility.TIME_COLUMN_NAME], y=df[single_column],
                        name=current_parameter_name, mode='lines',
                        line=dict(color=color_dict[current_parameter_name]),
                        showlegend=showlegend
                    ),
                    row=row, col=col
                )
    return fig


def _add_sensitivity_indices_as_heatmap(fig, df_temp, substring="generalized_sobol_total_index_", 
                                look_back_window_size=None, row=1, col=1, showscale=False, colorscale='Plasma'):
    """
    colorscale options - 'Plasma', 'Inferno', 'Viridis'
    """
    si_columns_to_plot = []
    for single_column in df_temp.columns:
        if single_column!=utility.TIME_COLUMN_NAME and single_column!='qoi' and single_column!='measured' and single_column.startswith(substring):
            if look_back_window_size is not None:
                current_look_back_window_size, current_parameter_name = identify_column_generlized_index_form(single_column)
                if current_look_back_window_size is not None and int(current_look_back_window_size) == look_back_window_size:
                    si_columns_to_plot.append(single_column)
            else:
                si_columns_to_plot.append(single_column)
    #si_columns_to_label = si_columns_to_label.reverse()
    df_generalized = df_temp[si_columns_to_plot].T
    # si_columns_to_label = [single_column.split('_')[-1] for single_column in si_columns_to_plot]
    si_columns_to_label = [extract_paramname(single_column, prefix=substring) for single_column in si_columns_to_plot]
    # print(f"DEBUGGING si_columns_to_label-{si_columns_to_label}")        
    trace=go.Heatmap(
        z=df_generalized[::-1],
        x=df_temp['TimeStamp'],
        y=si_columns_to_label[::-1],
        showscale=showscale,
        colorbar=dict(
            x=1.0,        # Position on the x-axis (1.0 is the far right, 1.1 moves it further right)
            y=0.5,        # Position on the y-axis (0.5 is centered vertically)
            len=0.35,     # Length of the colorbar as a fraction of the plot height
            thickness=20, # Thickness of the colorbar
            title='S.I. Scale' # Title of the colorbar
        ),
        colorscale=colorscale,
     )
    fig.add_trace(trace, row=row, col=col)
    return fig



def _add_e_std(fig, df_temp, name=f'E+-std', row=1, col=1, showlegend=True, transparency=0.4):
    if "E_minus_std" in df_temp.columns:
        fig.add_trace(go.Scatter(x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp["E_minus_std"],
                                 name=f'10th Percentile',
                                 line_color=f'rgba(128,128,128, {transparency})', mode='lines',
                                 showlegend=False,
                                ),
                      row=row, col=col)
    if "E_plus_std" in df_temp.columns:
        fig.add_trace(go.Scatter(x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp["E_plus_std"],
                                 name=name,
                                 mode='lines',
                                 line=dict(color=f'rgba(128,128,128, {transparency})'), fill='tonexty',
                                 fillcolor=f'rgba(128,128,128, {transparency})',
                                 showlegend=showlegend,
                                ),
                      row=row, col=col)
    return fig
    
def _add_10_90_percentiles(fig, df_temp, name=f'10th-90th Percentile', row=1, col=1, showlegend=True, transparency=0.4):
    if "P10" in df_temp.columns:
        fig.add_trace(go.Scatter(x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp["P10"],
                                 name=f'10th Percentile',
                                 line_color=f'rgba(128,128,128, {transparency})', mode='lines',
                                 showlegend=False,
                                ),
                      row=row, col=col)
    if "P90" in df_temp.columns:
        fig.add_trace(go.Scatter(x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp["P90"],
                                 name=name,
                                 mode='lines',
                                 line=dict(color=f'rgba(128,128,128, {transparency})'), fill='tonexty',
                                 fillcolor=f'rgba(128,128,128, {transparency})',
                                 showlegend=showlegend,
                                ),
                      row=row, col=col)
    return fig

# in utility this function is slightly different
# def _update_fig_with_si_data_for_single_df(
#     fig, current_df, plot_heatmap, si_columns_to_plot, si_columns_to_label, color_dict, current_row=1, current_col=1, showscale=False, showlegend=False):
# def _update_fig_with_si_data_for_single_df(fig, current_df, plot_heatmap, si_columns_to_plot, si_columns_to_label, current_row, color_dict, showscale=False, showlegend=False):
# colorscale='Viridis'
def _update_fig_with_si_data_for_single_df(
    fig, current_df, plot_heatmap, si_columns_to_plot, si_columns_to_label, 
    color_dict=COLORS_DICT, current_row=1, current_col=1, showscale=False, showlegend=False, colorscale='Plasma'):
    """
    colorscale options - 'Plasma', 'Inferno', 'Viridis'
    """
    if plot_heatmap:
        si_df_single_qoi_columns_to_plot = current_df[si_columns_to_plot].T
        trace=go.Heatmap(
                z=si_df_single_qoi_columns_to_plot,
                x=current_df[utility.TIME_COLUMN_NAME],
                y=si_columns_to_label,
                showscale=showscale,
                colorbar=dict(
                    x=1.0,        # Position on the x-axis (1.0 is the far right, 1.1 moves it further right)
                    y=0.5,        # Position on the y-axis (0.5 is centered vertically)
                    len=0.35,     # Length of the colorbar as a fraction of the plot height
                    thickness=20, # Thickness of the colorbar
                    title='S.I. Scale' # Title of the colorbar
                ),
                colorscale=colorscale
         )
        fig.add_trace(trace, row=current_row, col=current_col)
    else:
        si_df_single_qoi_columns_to_plot = current_df[si_columns_to_plot]
        for idx, single_column in enumerate(si_columns_to_plot):
            current_parameter_name = si_columns_to_label[idx] #single_column.split(substring, 1)[1]
            fig.add_trace(
                go.Scatter(
                    x=current_df[utility.TIME_COLUMN_NAME], y=current_df[single_column],
                    name=current_parameter_name, #text=current_parameter_name
                    mode='lines',
                    line=dict(color=color_dict[current_parameter_name]),
                    showlegend=showlegend
                ),
                row=current_row, col=current_col
            )  
    return fig    


def _update_fig_layout_and_save(fig, directory_for_saving_plots, fileName, \
                                timesteps_min, timesteps_max, plotting_generalized_indices=False, height=1400, width=1100,
                                white_template=False, save_fig=True):

    fig.update_layout(
        xaxis=dict(
            rangemode='normal',
            range=[timesteps_min, timesteps_max],
            type="date"
        ),
        # yaxis=dict(
        #     rangemode='normal',  # Ensures the range is not padded for markers
        #     autorange=True       # Auto-range is enabled
        # )
    )
    # if plotting_generalized_indices:
    #     fig.update_layout(
    #         legend=dict(orientation="h", yanchor="bottom", y=-0.04, xanchor="right", x=0.99),
    #         showlegend=True,
    #         # template="plotly_white",
    #     )

    fig.update_layout(
            # legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.8),
            showlegend=True,
            # template="plotly_white",
        )
        
    fig.update_xaxes(
        tickformat='%b %y',            # Format dates as "Month Day" (e.g., "Jan 01")
        dtick="M2"                     # Set tick interval to 1 day for denser ticks
    )

    fig.update_layout(height=height, width=width)
    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=20,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )

    # for current_row in range(0, n_rows):
    #     if current_row%2==0:
    #         fig.update_xaxes(visible=False, row=current_row+1, col=1)
        
    # for current_row in range(0, n_rows-1):
    #     fig.update_xaxes(visible=False, row=current_row+1, col=1)


    # fig.update_layout(
    # legend=dict(
    #         # orientation="h",
    #         x=0.28,            # 5% from the left side
    #         y=0.97,            # 95% from the bottom (top of plot area)
    #         xanchor='left',
    #         # xanchor='right', #'left'
    #         # x=1.05,            # 5% from the left side
    #         # y=1.00,            # 95% from the bottom (top of plot area)
    #         yanchor='top',
    #         bordercolor='gray',
    #         borderwidth=1
    #     ),
    # showlegend=True,
    # )

    if white_template:
        fig.update_layout(
            template='simple_white', #'plotly_white'
            #plot_bgcolor='white',  
        )
        fig.update_xaxes(showgrid=True, gridcolor='lightgray', gridwidth=1)
        fig.update_yaxes(showgrid=True, gridcolor='lightgray', gridwidth=1)

    if save_fig:
        fun_save_fig(fig, fileName, directory_for_saving_plots, height=height, width=width)
    return fig


def fun_save_fig(fig, fileName, directory_for_saving_plots, height=1400, width=1100):
    fileName_without_ext = pathlib.Path(fileName).stem
    plot_filename = str(directory_for_saving_plots) + "/" + f"{fileName_without_ext}.html"
    pyo.plot(fig, filename=plot_filename, auto_open=False)
    plot_filename = pathlib.Path(directory_for_saving_plots) / f"{fileName_without_ext}.pdf"
    fig.write_image(str(plot_filename), format="pdf", height=height, width=width)


# Paths

In [ ]:
# TODO - change these paths accordingly
# specific for HBV-SASK model
BASE_SOURCE_PATH = pathlib.Path.cwd().parents[0] # UQEF-Dynamic root
print(BASE_SOURCE_PATH)
hbv_model_data_path = (pathlib.Path.cwd() / ".." / "data" / "HBV-SASK-data").resolve()
hbv_model_data_path = BASE_SOURCE_PATH / "data" / "HBV-SASK-data"
print(f"hbv_model_data_path-{hbv_model_data_path}")
inputModelDir = hbv_model_data_path

basis_workingDir = hbv_model_data_path / "paper_uqef_dynamic_sim"
globa_directory_for_saving_plots = hbv_model_data_path /'oldman_2004_2007' / 'comparing_different_si_oldman_2004_2007'
if not str(globa_directory_for_saving_plots).endswith("/"):
    globa_directory_for_saving_plots = str(globa_directory_for_saving_plots) + "/"
globa_directory_for_saving_plots =  pathlib.Path(globa_directory_for_saving_plots)
globa_directory_for_saving_plots.mkdir(parents=True, exist_ok=True)  # create it if it doesn't exist

In [ ]:
set_lower_predictions_to_zero=True
set_mean_prediction_to_zero=True
correct_sobol_indices=False
read_saved_simulations=False 
read_saved_states=False
instantly_save_results_for_each_time_step=False  #W atchout, for some runs this might be True!
time_column_name=utility.TIME_COLUMN_NAME

* 7. gpce_p4_sgl6_ct07_2004_2007_oldman/
* (?) gpce_p5_sgl6_ct07_generalized_2004_2007_oldman/
* 8. gpce_p5_sgl7_ct07_2004_2007_oldman/
* 4. mc_10000_gpce4_ct07_lhc_2004_2007_oldman/
* 4.1 mc_gpce_p4_ct07_100000_lhc_nse02_2004_2007_oldman/ 
* 2. mc_150000_lhc_2004_2007_oldman/
*  3. mc_150000_lhc_nse02_2004_2007_oldman/
* 5. mc_gpce_p5_ct07_150000_random_2004_2007_oldman/
* 6. mc_gpce_p5_ct07_150000_random_nse02_2004_2007_oldman/
* 5.1(?) mc_gpce_p5_ct07_30000_lhc_2004_2007_oldman/
* 1. saltelli_240000_lhc_2004_2007_oldman/
* 9. mc_11d_pm_ar09_500000_lhc_2004_2007_oldman
* ----
* 10. 11D MC 500 000 Random Q_cms; AET Oldman (AET Sobol_m faulty) mc_11d_500000_random_2004_2007_oldman
* 11. 11D MC 500 000 Random NSE>0.2 Q_cms; AET Oldman mc_11d_500000_random_nse02_2004_2007_oldman
* 12. 11D MC 500 000 Random Autoregressive 0.9 Q_cms Oldman mc_11d_500000_random_autoregressive09_2004_2007_oldman
* 13. 11D MC gPCE p=4 CT0.7 500 000 Random Q_cms; Oldman mc_gpce_11d_p4_ct07_500000_random_2004_2007_oldman
* 14. 11D MC gPCE p=4 CT0.7 500 000 Random NSE>0.2 Q_cms; AET Oldman (alpha Sobol_m faulty) mc_gpce_11d_p4_ct07_500000_random_nse02_2004_2007_oldman
* 21. 11D MC gPCE p=4 CT0.7 20 000 Random Q_cms; AET Oldman mc_gpce_11d_p4_ct07_20000_random_2004_2007_oldman/
* 23. 11D MC gPCE p=4 CT0.7 20 000 LHC Q_cms; AET LARS mc_gpce_11d_p4_ct07_20000_lhc_lars_2004_2007_oldman/
* 19. 11D gPCE p=4 CT0.7 l=6 30000KPU Q_cms; AET gpce_11d_p4_ct07_l6_2004_2007_oldman/
* 17. 10D MC 500 000 Random Q_cms; AET Oldman mc_10d_500000_random_2004_2007_oldman/
* 18. 10D MC 500 000 Random NSE>0.2 Q_cms; AET Oldman mc_10d_500000_random_nse02_2004_2007_oldman/
* 15. 10D MC gPCE p=4 CT0.7 500 000 Random Q_cms Oldman mc_gpce_10d_p4_ct07_500000_random_2004_2007_oldman/
* 16. 10D MC gPCE p=4 CT0.7 500 000 Random NSE>0.2 Q_cms Oldman mc_gpce_10d_p4_ct07_500000_random_nse02_2004_2007_oldman/
* 20. 10D MC gPCE p=4 CT0.7 10 000 Random Q_cms; AET Oldman mc_gpce_10d_p4_ct07_10000_random_2004_2007_oldman/
* 24. 10D MC gPCE p=4 CT0.7 10 000 LHC Q_cms; AET LARS mc_gpce_10d_p4_ct07_10000_lhc_lars_2004_2007_oldman/
* 22. 10D gPCE p=4 CT0.7 l=6 19000KPU Q_cms; AET gpce_10d_p4_ct07_l6_2004_2007_oldman/



# Reading the ouput...

## 10. 11D MC 500 000 Random Q_cms; AET Oldman (AET Sobol_m faulty)

* 11D MC 500 000 Random 2004-05 Q_cms; AET Oldman hbv uq cm4.0196 
* 11D MC 500 000 Random 2005-06 Q_cms; AET Oldman hbv uq cm4.0195 
* 11D MC 500 000 Random 2006-07 Q_cms; AET Oldman 0.2 hbv uq cm4.0194
* mc_11d_500000_random_2004_2007_oldman

In [ ]:
workingDir_I = basis_workingDir / 'hbv_uq_cm4.0196'
workingDir_II = basis_workingDir / 'hbv_uq_cm4.0195'
workingDir_III = basis_workingDir / 'hbv_uq_cm4.0194'

directory_for_saving_plots = globa_directory_for_saving_plots / 'mc_11d_500000_random_2004_2007_oldman'

if not str(directory_for_saving_plots).endswith("/"):
    directory_for_saving_plots = str(directory_for_saving_plots) + "/"
directory_for_saving_plots =  pathlib.Path(directory_for_saving_plots)

instantly_save_results_for_each_time_step = False

statisticsObject_I, df_statistics_and_measured_I,  si_t_df_I, si_m_df_I, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_I, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step,
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)

statisticsObject_II, df_statistics_and_measured_II,  si_t_df_II, si_m_df_II, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_II, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step, 
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)

statisticsObject_III, df_statistics_and_measured_III,  si_t_df_III, si_m_df_III, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_III, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step, 
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)


df_statistics_and_measured_mc_11d = pd.concat([df_statistics_and_measured_I, df_statistics_and_measured_II, df_statistics_and_measured_III], ignore_index=True)
df_statistics_and_measured_mc_11d = df_statistics_and_measured_mc_11d.sort_values(by=utility.TIME_COLUMN_NAME)

timestamp_min = df_statistics_and_measured_mc_11d[utility.TIME_COLUMN_NAME].min()
timestamp_max = df_statistics_and_measured_mc_11d[utility.TIME_COLUMN_NAME].max()
print(f"timestamp_min={timestamp_min}; timestamp_max={timestamp_max}")

if si_m_df_I is not None and si_m_df_II is not None and si_m_df_III is not None:
    si_m_df_mc_11d = pd.concat([si_m_df_I, si_m_df_II, si_m_df_III], ignore_index=True)
    si_m_df_mc_11d = si_m_df_mc_11d.sort_values(by=utility.TIME_COLUMN_NAME)
else:
    si_m_df_mc_11d = None


if si_t_df_I is not None and si_t_df_II is not None and si_t_df_III is not None:
    si_t_df_mc_11d = pd.concat([si_t_df_I, si_t_df_II, si_t_df_III], ignore_index=True)
    si_t_df_mc_11d = si_t_df_mc_11d.sort_values(by=utility.TIME_COLUMN_NAME)
else:
    si_t_df_mc_11d = None

generalized_column_indices = [col for col in df_statistics_and_measured_mc_11d.columns if col.startswith("generalized_sobol_total_index_")]
columns_of_interes = generalized_column_indices + ['qoi', 'TimeStamp']
df_statistics_and_measured_generalized_mc_11 = df_statistics_and_measured_mc_11d[columns_of_interes]
# df_statistics_and_measured_generalized_mc_11

In [ ]:
# workingDir_I | workingDir_II | workingDir_III
nodes_file = workingDir_I / 'nodes.simnodes.zip'
with open(nodes_file, 'rb') as f:
    uqef_simulationNodes = pickle.load(f)
uqef_simulationNodes.distNodes.shape

In [ ]:
dict_with_results_of_interest = uqef_dynamic_utils.read_all_saved_uqef_dynamic_results_and_produce_dict_of_interest(
    workingDir=workingDir_I,
    read_saved_simulations=read_saved_simulations,
    read_saved_states=read_saved_states, 
)
print(f"dict_with_results_of_interest={dict_with_results_of_interest}")

## 11. 11D MC 500 000 Random NSE>0.2 Q_cms; AET Oldman

* 11D MC 500 000 Random NSE>0.2 2004-05 Q_cms; AET Oldman hbv uq cm4.0211 
* 11D MC 500 000 Random NSE>0.2 2004-05 Q_cms; AET Oldman hbv uq cm4.0212 
* 11D MC 500 000 Random NSE>0.2 2004-05 Q_cms; AET Oldman hbv uq cm4.0213
* mc_11d_500000_random_nse02_2004_2007_oldman

In [ ]:
workingDir_I = basis_workingDir / 'hbv_uq_cm4.0211'
workingDir_II = basis_workingDir / 'hbv_uq_cm4.0212'
workingDir_III = basis_workingDir / 'hbv_uq_cm4.0213'

directory_for_saving_plots = globa_directory_for_saving_plots / 'mc_11d_500000_random_2004_2007_oldman'

if not str(directory_for_saving_plots).endswith("/"):
    directory_for_saving_plots = str(directory_for_saving_plots) + "/"
directory_for_saving_plots =  pathlib.Path(directory_for_saving_plots)

instantly_save_results_for_each_time_step = False

statisticsObject_I, df_statistics_and_measured_I,  si_t_df_I, si_m_df_I, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_I, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step,
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)

statisticsObject_II, df_statistics_and_measured_II,  si_t_df_II, si_m_df_II, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_II, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step, 
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)

statisticsObject_III, df_statistics_and_measured_III,  si_t_df_III, si_m_df_III, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_III, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step, 
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)


df_statistics_and_measured_mc_11d_nse02 = pd.concat([df_statistics_and_measured_I, df_statistics_and_measured_II, df_statistics_and_measured_III], ignore_index=True)
df_statistics_and_measured_mc_11d_nse02 = df_statistics_and_measured_mc_11d_nse02.sort_values(by=utility.TIME_COLUMN_NAME)

timestamp_min = df_statistics_and_measured_mc_11d_nse02[utility.TIME_COLUMN_NAME].min()
timestamp_max = df_statistics_and_measured_mc_11d_nse02[utility.TIME_COLUMN_NAME].max()
print(f"timestamp_min={timestamp_min}; timestamp_max={timestamp_max}")

if si_m_df_I is not None and si_m_df_II is not None and si_m_df_III is not None:
    si_m_df_mc_11d_nse02 = pd.concat([si_m_df_I, si_m_df_II, si_m_df_III], ignore_index=True)
    si_m_df_mc_11d_nse02 = si_m_df_mc_11d_nse02.sort_values(by=utility.TIME_COLUMN_NAME)
else:
    si_m_df_mc_11d_nse02 = None


if si_t_df_I is not None and si_t_df_II is not None and si_t_df_III is not None:
    si_t_df_mc_11d_nse02 = pd.concat([si_t_df_I, si_t_df_II, si_t_df_III], ignore_index=True)
    si_t_df_mc_11d_nse02 = si_t_df_mc_11d_nse02.sort_values(by=utility.TIME_COLUMN_NAME)
else:
    si_t_df_mc_11d_nse02 = None

generalized_column_indices = [col for col in df_statistics_and_measured_mc_11d_nse02.columns if col.startswith("generalized_sobol_total_index_")]
columns_of_interes = generalized_column_indices + ['qoi', 'TimeStamp']
df_statistics_and_measured_generalized_mc_11_nse02 = df_statistics_and_measured_mc_11d_nse02[columns_of_interes]
# df_statistics_and_measured_generalized_mc_11_nse02

In [ ]:
df_index_parameter_conditioned_file = workingDir_I / utility.DF_INDEX_PARAMETER_CONDITIONED_FILE
if df_index_parameter_conditioned_file.is_file():
    df_index_parameter_conditioned = pd.read_pickle(df_index_parameter_conditioned_file, compression="gzip")
df_index_parameter_conditioned

## 17. 10D MC 500 000 Random Q_cms; AET Oldman

* 10D MC 500 000 Random 2004-05 Q_cms; AET Oldman hbv uq cm4.0264 
* 10D MC 500 000 Random 2004-05 Q_cms; AET Oldman hbv uq cm4.0265 
* 10D MC 500 000 Random 2004-05 Q_cms; AET Oldman hbv uq cm4.0266
* mc_10d_500000_random_2004_2007_oldman

In [ ]:
workingDir_I = basis_workingDir / 'hbv_uq_cm4.0264'
workingDir_II = basis_workingDir / 'hbv_uq_cm4.0265'
workingDir_III = basis_workingDir / 'hbv_uq_cm4.0266'

directory_for_saving_plots = globa_directory_for_saving_plots / 'mc_11d_500000_random_2004_2007_oldman'

if not str(directory_for_saving_plots).endswith("/"):
    directory_for_saving_plots = str(directory_for_saving_plots) + "/"
directory_for_saving_plots =  pathlib.Path(directory_for_saving_plots)

instantly_save_results_for_each_time_step = False

statisticsObject_I, df_statistics_and_measured_I,  si_t_df_I, si_m_df_I, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_I, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step,
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)

statisticsObject_II, df_statistics_and_measured_II,  si_t_df_II, si_m_df_II, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_II, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step, 
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)

statisticsObject_III, df_statistics_and_measured_III,  si_t_df_III, si_m_df_III, _, _, _, _, simulationNodes= \
create_stat_object.get_df_statistics_and_df_si_from_saved_files(
    workingDir_III, inputModelDir, 
    set_lower_predictions_to_zero=set_lower_predictions_to_zero,
    set_mean_prediction_to_zero=set_mean_prediction_to_zero,
    correct_sobol_indices=correct_sobol_indices,
    read_saved_simulations=read_saved_simulations, 
    read_saved_states=read_saved_states, 
    instantly_save_results_for_each_time_step=instantly_save_results_for_each_time_step, 
    add_measured_data=True,
    add_forcing_data=True,
    transform_measured_data_as_original_model=True,
)


df_statistics_and_measured_mc_10d = pd.concat([df_statistics_and_measured_I, df_statistics_and_measured_II, df_statistics_and_measured_III], ignore_index=True)
df_statistics_and_measured_mc_10d = df_statistics_and_measured_mc_10d.sort_values(by=utility.TIME_COLUMN_NAME)

timestamp_min = df_statistics_and_measured_mc_10d[utility.TIME_COLUMN_NAME].min()
timestamp_max = df_statistics_and_measured_mc_10d[utility.TIME_COLUMN_NAME].max()
print(f"timestamp_min={timestamp_min}; timestamp_max={timestamp_max}")

if si_m_df_I is not None and si_m_df_II is not None and si_m_df_III is not None:
    si_m_df_mc_10d = pd.concat([si_m_df_I, si_m_df_II, si_m_df_III], ignore_index=True)
    si_m_df_mc_10d = si_m_df_mc_10d.sort_values(by=utility.TIME_COLUMN_NAME)
else:
    si_m_df_mc_10d = None


if si_t_df_I is not None and si_t_df_II is not None and si_t_df_III is not None:
    si_t_df_mc_10d = pd.concat([si_t_df_I, si_t_df_II, si_t_df_III], ignore_index=True)
    si_t_df_mc_10d = si_t_df_mc_10d.sort_values(by=utility.TIME_COLUMN_NAME)
else:
    si_t_df_mc_10d = None

generalized_column_indices = [col for col in df_statistics_and_measured_mc_10d.columns if col.startswith("generalized_sobol_total_index_")]
columns_of_interes = generalized_column_indices + ['qoi', 'TimeStamp']
df_statistics_and_measured_generalized_mc_10 = df_statistics_and_measured_mc_10d[columns_of_interes]
# df_statistics_and_measured_generalized_mc_11

# Plotting

In [ ]:
# df_statistics_and_measured_mc_10d
# df_statistics_and_measured_mc_10d_nse02
# df_statistics_and_measured_mc_gpce_p4_10d_500000_random
# df_statistics_and_measured_mc_gpce_p4_10d_500000_random_nse02

# Comparing PCE (MC) vs Simple MC in 10D
single_qoi = "Q_cms"  # AET | "Q_cms"

plot_forcing_data = True

df_statistics_and_measured_mc_11d_subset = df_statistics_and_measured_mc_11d.loc[\
    df_statistics_and_measured_mc_11d[utility.QOI_ENTRY]==single_qoi]

df_statistics_and_measured_mc_11d_nse02_subset = df_statistics_and_measured_mc_11d_nse02.loc[\
df_statistics_and_measured_mc_11d_nse02[utility.QOI_ENTRY]==single_qoi]

df_statistics_and_measured_mc_10d_subset = df_statistics_and_measured_mc_10d.loc[\
    df_statistics_and_measured_mc_10d[utility.QOI_ENTRY]==single_qoi]

# df_statistics_and_measured_mc_10d_nse02_subset = df_statistics_and_measured_mc_10d_nse02.loc[\
# df_statistics_and_measured_mc_10d_nse02[utility.QOI_ENTRY]==single_qoi]

methods = {
    "MC 11D (500k random)":           df_statistics_and_measured_mc_11d_subset,
    "MC 11D (500k, NSE > 0.2)":       df_statistics_and_measured_mc_11d_nse02_subset,
    "MC 10D (500k random)":           df_statistics_and_measured_mc_10d_subset,
}
# Colorblind-friendly palette (Wong 2011)
method_colors = ["#E69F00", "#56B4E9", "#009E73"]

name = f'10th-90th Percentile'
# name = f'E+-std'

n_rows = 4
starting_row = 1
subplot_titles = [
    "FUQ: QoI - Streamflow[m^3/s] - MC 11D 500,000 Random Samples", 
    "FUQ: QoI - Streamflow[m^3/s] - MC 11D 500,000 Random Samples Filtered based on metric NSE > 0.2",
    "FUQ: QoI - Streamflow[m^3/s] - MC 10D 500,000 Random Samples", 
    "Relative band width: (P90 − P10) / measured  [−]",
    # "FUQ: QoI - Streamflow[m^3/s] - MC 10D 500,000 Random Samples Filtered based on metric NSE > 0.2",
]


if plot_forcing_data:
    n_rows += 2
    starting_row = 3
    subplot_titles = ["Temperature [°C]", "Precipitation [mm/day]",] + subplot_titles


fig = make_subplots(
    rows=n_rows, cols=1,
    subplot_titles=subplot_titles,
    shared_xaxes=False,
    vertical_spacing=0.04,
)

helper_df = df_statistics_and_measured_mc_11d_subset

timesteps_min = helper_df[utility.TIME_COLUMN_NAME].min()
timesteps_max = helper_df[utility.TIME_COLUMN_NAME].max()

if plot_forcing_data:
    fig = _add_forcing_data(fig, helper_df)

current_row = starting_row
showlegend = True

####### 1 ########
fig.add_trace(
    go.Scatter(
        x=helper_df[utility.TIME_COLUMN_NAME], y=helper_df['measured'],
        name="Measured Streamflow [m^3/s]", mode='lines',
        # line=dict(color='green'),
        # line=dict(color="black", width=1.5),
        line=dict(color="#CC0000", width=2),
        # line=dict(color="#CC0000", width=2, dash="dot"),
        showlegend=showlegend
    ),
    row=current_row, col=1
)
df_temp = df_statistics_and_measured_mc_11d_subset
fig.add_trace(
    go.Scatter(
        x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp['E'],
        text=df_temp['E'],
        name=f"Mean predicted Streamflow [m^3/s]", mode='lines',
        line=dict(color='#0072B2'),
        showlegend=showlegend
    ),
    row=current_row, col=1
)
# fig = _add_e_std(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
# fig = _add_e_2std(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
fig = _add_10_90_percentiles(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
current_row += 1

###############

showlegend = False

####### 2 ########
fig.add_trace(
    go.Scatter(
        x=helper_df[utility.TIME_COLUMN_NAME], y=helper_df['measured'],
        name="Measured Streamflow [m^3/s]", mode='lines',
        # line=dict(color='green'),
        # line=dict(color="black", width=1.5),
        line=dict(color="#CC0000", width=2),
        # line=dict(color="#CC0000", width=2, dash="dot"),
        showlegend=showlegend
    ),
    row=current_row, col=1
)
df_temp = df_statistics_and_measured_mc_11d_nse02_subset
fig.add_trace(
    go.Scatter(
        x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp['E'],
        text=df_temp['E'],
        name=f"Mean predicted Streamflow [m^3/s]", mode='lines',
        line=dict(color='#0072B2'),
        showlegend=showlegend
    ),
    row=current_row, col=1
)
# fig = _add_e_std(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
# fig = _add_e_2std(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
fig = _add_10_90_percentiles(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
current_row += 1

####### 3 ########
fig.add_trace(
    go.Scatter(
        x=helper_df[utility.TIME_COLUMN_NAME], y=helper_df['measured'],
        name="Measured Streamflow [m^3/s]", mode='lines',
        # line=dict(color='green'),
        # line=dict(color="black", width=1.5),
        line=dict(color="#CC0000", width=2),
        # line=dict(color="#CC0000", width=2, dash="dot"),
        showlegend=showlegend
    ),
    row=current_row, col=1
)
df_temp = df_statistics_and_measured_mc_10d_subset
fig.add_trace(
    go.Scatter(
        x=df_temp[utility.TIME_COLUMN_NAME], y=df_temp['E'],
        text=df_temp['E'],
        name=f"Mean predicted Streamflow [m^3/s]", mode='lines',
        line=dict(color='#0072B2'),
        showlegend=showlegend
    ),
    row=current_row, col=1
)
# fig = _add_e_std(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
# fig = _add_e_2std(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
fig = _add_10_90_percentiles(fig, df_temp, name=name, row=current_row, col=1, showlegend=showlegend)
current_row += 1

####### Relative band ########

for (label, df), color in zip(methods.items(), method_colors):
    bw_abs  = df["P90"] - df["P10"]
    bw_rel  = bw_abs / df["measured"].replace(0, float("nan"))

    fig.add_trace(
        go.Scatter(x=df[utility.TIME_COLUMN_NAME], y=bw_rel, name=label,
                   mode="lines", line=dict(color=color), showlegend=True),
        row=current_row, col=1
    )

####### Saving ########
fig.update_yaxes(title_text="T [°C]",       row=1, col=1)
fig.update_yaxes(title_text="P [mm/day]",   row=2, col=1)
fig.update_yaxes(title_text="Q [m³/s]",     row=3, col=1)
fig.update_yaxes(title_text="Q [m³/s]",     row=4, col=1)
fig.update_yaxes(title_text="Q [m³/s]",     row=5, col=1)
fig.update_yaxes(title_text="(P90−P10)/obs [−]", row=n_rows, col=1)
# fig.update_layout(height=900, width=1100)
# fig.update_layout(height=750, width=1100)
fig.update_layout(
        # legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0), #x=0.99
        showlegend=True,
        # template="plotly_white", #"simple_white"
        # margin=dict(t=60, b=40, l=60, r=20),
    )
fig.update_xaxes(
    tickformat='%b %y',            # Format dates as "Month Day" (e.g., "Jan 01")
    dtick="M2"                     # Set tick interval to 1 day for denser ticks
)
fileName = "UQ_Q_cms_11d_and_10d_mc_filtering_and_nofiltering_shorten_forcing.html"
fig = _update_fig_layout_and_save(
    fig=fig, directory_for_saving_plots=globa_directory_for_saving_plots, fileName=fileName, \
    timesteps_min=timesteps_min, timesteps_max=timesteps_max, \
    save_fig=True, plotting_generalized_indices=False, width=1000, height=1000)

fig.show()

## Alternative: Explicit Uncertainty Band Width Comparison

Reviewer suggestion: instead of visually comparing grey band widths across separate subplots, we directly plot the **band width** (P90 − P10) for all three methods on the **same axes**. This makes quantitative comparison unambiguous.

Two complementary views:
1. **Absolute band width** (P90 − P10) over time — all methods overlaid
2. **Relative band width** ((P90 − P10) / measured) — normalises for flow magnitude, useful since wide bands during high-flow events can be trivially expected

In [ ]:
# --- Method labels and corresponding dataframes ---
methods = {
    "MC 11D (500k random)":      df_statistics_and_measured_mc_11d_subset,
    "MC 11D (500k, NSE > 0.2)":  df_statistics_and_measured_mc_11d_nse02_subset,
    "MC 10D (500k random)":      df_statistics_and_measured_mc_10d_subset,
}
# Colorblind-friendly palette (Wong 2011)
method_colors = ["#E69F00", "#56B4E9", "#009E73"]

# --- Compute band widths ---
for label, df in methods.items():
    assert "P10" in df.columns and "P90" in df.columns, f"P10/P90 missing for {label}"

time_col = utility.TIME_COLUMN_NAME

fig_bw = make_subplots(
    rows=3, cols=1,
    subplot_titles=[
        "Absolute band width: P90 − P10  [m³/s]",
        "Relative band width: (P90 − P10) / measured  [−]",
        "Measured streamflow  [m³/s]  (reference)",
    ],
    shared_xaxes=True,
    vertical_spacing=0.08,
)

for (label, df), color in zip(methods.items(), method_colors):
    bw_abs = df["P90"] - df["P10"]
    bw_rel = bw_abs / df["measured"].replace(0, float("nan"))

    fig_bw.add_trace(
        go.Scatter(x=df[time_col], y=bw_abs, name=label,
                   mode="lines", line=dict(color=color)),
        row=1, col=1
    )
    fig_bw.add_trace(
        go.Scatter(x=df[time_col], y=bw_rel, name=label,
                   mode="lines", line=dict(color=color), showlegend=False),
        row=2, col=1
    )

ref_df = df_statistics_and_measured_mc_11d_subset
fig_bw.add_trace(
    go.Scatter(x=ref_df[time_col], y=ref_df["measured"],
               name="Measured", mode="lines",
               line=dict(color="green"), showlegend=True),
    row=3, col=1
)

fig_bw.update_layout(
    height=750, width=1100,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.99),
    template="simple_white",
    margin=dict(t=60, b=40, l=60, r=20),
)
fig_bw.update_xaxes(tickformat="%b %y")
fig_bw.update_yaxes(title_text="P90−P10 [m³/s]",    row=1, col=1)
fig_bw.update_yaxes(title_text="(P90−P10)/obs [−]", row=2, col=1)
fig_bw.update_yaxes(title_text="Q [m³/s]",          row=3, col=1)

fig_bw.write_image(str(globa_directory_for_saving_plots / "band_width_comparison.pdf"),
                   format="pdf", height=750, width=1100)
fig_bw.show()

In [ ]:
# --- Summary statistics: box plots of absolute band width per method ---
fig_box = go.Figure()
for (label, df), color in zip(methods.items(), method_colors):
    bw_abs = (df["P90"] - df["P10"]).dropna()
    fig_box.add_trace(go.Box(
        y=bw_abs,
        name=label,
        marker_color=color,
        boxmean="sd",
        boxpoints=False,
    ))

fig_box.update_layout(
    title="Distribution of absolute band width (P90 − P10) per UQ method",
    yaxis_title="P90 − P10  [m³/s]",
    template="simple_white",
    height=500, width=700,
    showlegend=False,
)

fig_box.write_image(str(globa_directory_for_saving_plots / "band_width_boxplot.pdf"),
                    format="pdf", height=500, width=700)
fig_box.show()

# --- Print summary table ---
import pandas as pd
rows = []
for label, df in methods.items():
    bw = (df["P90"] - df["P10"]).dropna()
    rows.append({
        "Method": label,
        "Mean [m³/s]":   round(bw.mean(), 3),
        "Median [m³/s]": round(bw.median(), 3),
        "Std [m³/s]":    round(bw.std(), 3),
        "Max [m³/s]":    round(bw.max(), 3),
    })
pd.DataFrame(rows).set_index("Method")

In [ ]:
# Overlay all three P10-P90 bands on a single streamflow panel
# with distinct colors + dashed band boundaries so they remain distinguishable

band_fills   = ["rgba(230,159,0,0.25)",  "rgba(86,180,233,0.25)",  "rgba(0,158,115,0.25)"]
band_borders = ["rgba(230,159,0,0.80)",  "rgba(86,180,233,0.80)",  "rgba(0,158,115,0.80)"]

ref_df   = df_statistics_and_measured_mc_11d_subset
time_col = utility.TIME_COLUMN_NAME

fig_ov = make_subplots(
    rows=3, cols=1,
    subplot_titles=["Temperature [°C]", "Precipitation [mm/day]",
                    "Streamflow [m³/s] — all UQ methods overlaid"],
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.15, 0.15, 0.70],
)

# --- Forcing data ---
fig_ov.add_trace(
    go.Scatter(x=ref_df[time_col], y=ref_df["temperature"],
               name="Temperature", mode="lines",
               line=dict(color="steelblue"), showlegend=False),
    row=1, col=1,
)
fig_ov.add_trace(
    go.Scatter(x=ref_df[time_col], y=ref_df["precipitation"],
               name="Precipitation", mode="lines",
               line=dict(color="#CC79A7"), showlegend=False),
    row=2, col=1,
)

# --- P10-P90 bands, one per method ---
for (label, df), fill, border in zip(methods.items(), band_fills, band_borders):
    fig_ov.add_trace(
        go.Scatter(
            x=df[time_col], y=df["P10"],
            mode="lines",
            line=dict(color=border, width=0.8, dash="dash"),
            showlegend=False,
        ),
        row=3, col=1,
    )
    fig_ov.add_trace(
        go.Scatter(
            x=df[time_col], y=df["P90"],
            name=f"{label}  (P10–P90)",
            mode="lines",
            line=dict(color=border, width=0.8, dash="dash"),
            fill="tonexty",
            fillcolor=fill,
            showlegend=True,
        ),
        row=3, col=1,
    )

# --- Measured streamflow on top ---
fig_ov.add_trace(
    go.Scatter(
        x=ref_df[time_col], y=ref_df["measured"],
        name="Measured streamflow",
        mode="lines",
        line=dict(color="black", width=1.5),
    ),
    row=3, col=1,
)

fig_ov.update_layout(
    height=800, width=1100,
    template="simple_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.99),
    margin=dict(t=60, b=40, l=60, r=20),
)
fig_ov.update_xaxes(tickformat="%b %y")
fig_ov.update_yaxes(title_text="T [°C]",     row=1, col=1)
fig_ov.update_yaxes(title_text="P [mm/day]", row=2, col=1)
fig_ov.update_yaxes(title_text="Q [m³/s]",  row=3, col=1)

fig_ov.write_image(str(globa_directory_for_saving_plots / "overlaid_bands_temp_prec_streamflow.pdf"),
                   format="pdf", height=800, width=1100)
fig_ov.show()